# Next-Concept Modeling for 7×7 Queens

Train one NCM run to solve a **given region coloring** by predicting a hierarchy of solution codes. Queen cells retain their original colors; all other cells become EMPTY. The coloring is provided as a fixed prefix condition, following the puzzle prefix used for Sudoku in NCM.

Based on [Next-Concept Modeling](https://arxiv.org/abs/2610.14162), October 9, 2026; [official implementation](https://github.com/HazyResearch/ncm/tree/c70274273f489abc73832b2b98584a8483b6b203).

This is a compact **2D adaptation**: residual VQ + nested dropout → frozen tokenizer → next-level transformer. No pretrained text weights are used. The combined tokenizer and generator capacity is close to the existing Queens DFM U-Net; training uses a larger explicit update budget to allow convergence.

Retain the existing train/validation/test split, augmentations, 256 test puzzles, and 32 independent attempts. No solution search or constraint repair is used. Code sampling is stochastic, so retry curves retain their usual interpretation.

Both tokenizer and generator train from scratch for their full budgets. Set `RESUME_CHECKPOINT = True` only to resume this new run after interruption.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import hashlib
import html
import json
import math
import numpy as np
import torch
import plotly.graph_objects as go
from IPython.display import HTML, display
from torch.nn import functional as F
from torch.utils.data import Dataset, TensorDataset
from _shared.model import Denoiser

import importlib
import _shared.ncm as ncm_module
import _shared.ncm_training as ncm_training_module
import _shared.ncm_benchmarks as ncm_benchmarks_module

importlib.reload(ncm_module)
importlib.reload(ncm_training_module)
importlib.reload(ncm_benchmarks_module)

from _shared.ncm import NCMConfig, GridNCM, matched_config, PAPER, REFERENCE
from _shared.ncm_training import NCMRun, measure_baseline, train_online, file_sha256
from _shared.ncm_benchmarks import board_panels, PLOT_CONFIG, progress_bar

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from _shared.queens_data import (
    BOARD_SIZE,
    EMPTY,
    load_queens_dataset,
    split_puzzles,
    solution_checks,
)
from _shared.queens_dfm_inference import QueensDFM, load_queens_dfm, sample_queens_dfm
from _shared.ncm_benchmarks import evaluate_queens, plot_queens_retries

SEED = 17
torch.manual_seed(SEED)
np.random.seed(SEED)
ARTIFACTS = PROJECT_ROOT / "artifacts" / "queens7" / "ncm"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACTS / "queens7_ncm.pt"
DFM_CHECKPOINT_PATH = (
    PROJECT_ROOT / "artifacts" / "queens7" / "queens7_discrete_flow_matching.pt"
)

## 1. Data and the existing test split

Load the prepared `data/queens7` collection; no new puzzles are generated here. Split before augmentation, as in DFM, with seed 17. Training applies paired random rotations/reflections and color-ID permutations; validation and test remain unchanged.

Each row, column, and color must contain one queen. Queens cannot touch diagonally adjacent cells; longer diagonals are allowed.


In [2]:
puzzles = load_queens_dataset(PROJECT_ROOT / "data" / "queens7")
splits = split_puzzles(puzzles["fingerprints"], seed=SEED)
regions, queen_masks = puzzles["regions"], puzzles["queens"]
target_tokens = np.where(queen_masks, regions, EMPTY).astype(np.int64)


class QueensDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices, self.augment = np.asarray(indices), augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = int(self.indices[item])
        board = torch.from_numpy(regions[index].astype(np.int64))
        queens = torch.from_numpy(queen_masks[index].copy())
        if self.augment:
            turns = int(torch.randint(4, ()))
            board, queens = torch.rot90(board, turns, (-2, -1)), torch.rot90(
                queens, turns, (-2, -1)
            )
            if bool(torch.randint(2, ())):
                board, queens = board.flip(-1), queens.flip(-1)
            board = torch.randperm(BOARD_SIZE)[board]
        return board.contiguous(), torch.where(queens, board, EMPTY).contiguous()


train_dataset = QueensDataset(splits["train"], augment=True)
validation_dataset = QueensDataset(splits["validation"])
data_signature = dict(
    task="queens7",
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    splits={name: values.tolist() for name, values in splits.items()},
    seed=SEED,
)
display(
    HTML(
        f"<b>{len(regions):,} puzzles</b> · "
        + " · ".join(f"{name}: {len(values):,}" for name, values in splits.items())
    )
)

In [3]:
example_rng = np.random.default_rng(SEED + 201)
for source in sorted(set(puzzles["source_names"].tolist())):
    available = np.flatnonzero(puzzles["source_names"] == source)
    chosen = example_rng.choice(available, min(2, len(available)), replace=False)
    board_panels(
        regions[chosen],
        [f"Example {j+1}" for j in range(len(chosen))],
        f"{source} · {len(available)} puzzles",
        colorings=regions[chosen],
        queens=True,
        queen_masks=queen_masks[chosen],
        columns=2,
    ).show(config=PLOT_CONFIG)

## 2. Architecture and conditioning

The tokenizer learns to reconstruct the **target** grid using 8 classes: 7 colors + EMPTY. Scales: **1×1 → 2×2 → 3×3 → 4×4 → 5×5 → 7×7**. As in the authors' Sudoku configuration, each level has only **16 codes**; latent width is reduced to 64 to fit our budget.

The generator receives the 49 original color tokens as a prefix with their 2D positions. Prefix tokens attend only to the prefix; target positions at the current level attend to the whole prefix and their own level. Future levels and the current ground-truth codes are unavailable. Color embeddings come from the frozen tokenizer; the projection into the generator width is trainable.

Global color/position and code weights are shared. Width and transformer depth are selected automatically by the **total** tokenizer + decoder + codebooks + generator parameter count, close to the DFM U-Net's 4,327,560. Model size is not selected using test solvability.


In [4]:
BATCH_SIZE = 128
MICRO_BATCH_SIZE = 32
SAMPLES_PER_EPOCH = 4096
with torch.device("meta"):
    baseline_parameters = sum(p.numel() for p in QueensDFM().parameters())
proposal = NCMConfig(
    size=7, classes=8, scales=(1, 2, 3, 4, 5, 7), conditional=True, codebook_size=16
)
config, ncm_parameters = matched_config(proposal, baseline_parameters)
run = NCMRun(
    config,
    CHECKPOINT_PATH,
    data_signature=data_signature,
    seed=SEED,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    micro_batch_size=MICRO_BATCH_SIZE,
)
counts = run.model.parameter_counts()
display(
    HTML(
        f"<b>NCM:</b> {counts['total']:,} = tokenizer {counts['tokenizer']:,} + generator {counts['generator']:,}<br>"
        f"<b>DFM:</b> {baseline_parameters:,} · difference {100*(counts['total']/baseline_parameters-1):+.2f}%<br>"
        f"Generator: width {config.width}, {config.layers} blocks · expanded training budget"
    )
)

### Fresh run or explicit resume

By default, both the tokenizer and generator start from random initialization.
Set `RESUME_CHECKPOINT = True` only to resume a checkpoint from this new run after interruption.
Run setup and model initialization before this cell. Loading restores both stages,
optimizer states, RNG state, and history; it does not shorten the tokenizer's target.


In [5]:
RESUME_CHECKPOINT = False
if RESUME_CHECKPOINT:
    if not CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            "No NCM checkpoint to resume. Use RESUME_CHECKPOINT = False for a fresh run."
        )
    run.load()
    display(
        HTML(
            "<b>Resuming NCM:</b> "
            + " · ".join(
                f"{stage}: {state['updates']:,} updates"
                for stage, state in run.state.items()
            )
        )
    )
else:
    if any(state["updates"] for state in run.state.values()):
        raise RuntimeError(
            "Re-run the model initialization cell to start both stages from scratch."
        )
    display(
        HTML("<b>Fresh NCM run:</b> tokenizer and generator initialized from scratch.")
    )

## 3. Full training budgets for both stages

Train **both stages from scratch**. Stage I always runs its full update target,
even if validation pixel or whole-grid accuracy reaches 100% early. Reconstruction
accuracy alone is not used to skip training or freeze an unfinished tokenizer.

Defaults: **mazes — 1,000 tokenizer + 2,000 generator updates**;
**Queens — 2,000 tokenizer + 20,000 generator updates**. Targets are total optimizer
updates for this run. These expanded budgets do not claim equal training time with
earlier baselines. Actual elapsed time and data exposure are recorded.

Stage II is allowed only after the complete Stage I budget has been trained.
Then check exact whole-grid reconstruction on 128 validation examples and freeze
the tokenizer. If it fails the readiness check, extend Stage I before starting
the generator. The hierarchy cannot be changed after generator training begins.

The generator uses BF16 on supported CUDA devices, otherwise FP32. Tokenizer,
code assignment, and decoder remain FP32. Warmup precedes cosine decay over the
configured update count. Explicit resume continues these same full targets.


In [6]:
TOKENIZER_TARGET_UPDATES = 2000
GENERATOR_TARGET_UPDATES = 20000
TOKENIZER_MIN_EXACT = 0.99
GENERATOR_PRECISION = "auto"
PACK_LEVELS = True
LEARNING_RATE = 2e-4

run.micro_batch_size = MICRO_BATCH_SIZE
run.learning_rate = LEARNING_RATE
run.set_training_targets(
    tokenizer_updates=TOKENIZER_TARGET_UPDATES,
    generator_updates=GENERATOR_TARGET_UPDATES,
    precision=GENERATOR_PRECISION,
    pack_levels=PACK_LEVELS,
    warmup_updates=100,
)
display(
    HTML(
        f"<b>Full update targets:</b> tokenizer {TOKENIZER_TARGET_UPDATES:,}; generator {GENERATOR_TARGET_UPDATES:,}<br>"
        f"Completed: tokenizer {run.state['tokenizer']['updates']:,}; generator {run.state['generator']['updates']:,}<br>"
        f"Generator precision: {run.model.generator_precision} · effective batch {run.batch_size} · microbatch {run.micro_batch_size}"
    )
)

## 4. Stage I — learn the code hierarchy

For a target grid $x$, the encoder produces $z=E_\phi(x)$. At each scale, quantize the residual left by the preceding levels:

$$
r_k=\operatorname{VQ}_k\!\left(\operatorname{Down}_k(z-\widehat z^{(k-1)})\right),
\qquad
\widehat z^{(k)}=\widehat z^{(k-1)}+R_k(e_k(r_k)).
$$

With probability $0.5$, retain a random incomplete prefix of levels; otherwise, retain all levels. The loss at the selected cutoff $k$ follows Equation (9) of the paper:

$$
\mathcal L_{\mathrm{tok}}=
\operatorname{CE}(x,D_\psi(\widehat z^{(k)}))
+0.25\,\operatorname{MSE}(z,\operatorname{sg}[\widehat z^{(k)}])
+\operatorname{MSE}(\operatorname{sg}[z],\widehat z^{(k)}).
$$

A straight-through estimator handles discrete selection; codebooks use EMA updates, following the authors' implementation. The learned first-level operator preserves global information, while residual convolutions after upsampling add detail. Validation always uses **all** levels, so its objective can differ from training with nested dropout.


In [7]:
import importlib
import _shared.ncm_training as ncm_training

train_online = importlib.reload(ncm_training).train_online

if run.state["tokenizer"]["updates"] < run.plan["tokenizer_updates"]:
    train_online(run, "tokenizer", train_dataset, validation_dataset, report_every=50)
else:
    display(HTML("The full tokenizer update target has been completed."))

### 4.1 Hierarchical solution reconstruction

Show how a **known validation target** is reconstructed level by level. This directly encodes the answer rather than solving the coloring. The tokenizer never receives test answers during training.

For 128 validation puzzles, separately measure pixel accuracy and exact reconstruction. Structural validation decodes without a color mask: a lost color or one incorrect cell can already invalidate a solution. This diagnoses the tokenizer's reconstruction ceiling.


In [8]:
if run.state["tokenizer"]["updates"] == 0:
    raise RuntimeError("Train or load the tokenizer first.")
run.model.eval()
index = int(splits["validation"][0])
with torch.no_grad():
    example = torch.from_numpy(target_tokens[index : index + 1]).to(DEVICE)
    _, partial = run.model.tokenizer.quantize(run.model.tokenizer.latent(example))
    reconstructions = [
        run.model.tokenizer.decode(z).argmax(1)[0].cpu().numpy() for z in partial
    ]
    board_panels(
        [regions[index], *reconstructions, target_tokens[index]],
        [
            "Puzzle",
            *[f"Level {i+1}: {s}×{s}" for i, s in enumerate(config.scales)],
            "Target",
        ],
        "NCM tokenizer · known solution reconstruction",
        colorings=[regions[index]] * (len(reconstructions) + 2),
        columns=4,
    ).show(config=PLOT_CONFIG)
    validation_indices = splits["validation"][:128]
    restored = []
    for start in range(0, len(validation_indices), MICRO_BATCH_SIZE):
        target = torch.from_numpy(
            target_tokens[validation_indices[start : start + MICRO_BATCH_SIZE]]
        ).to(DEVICE)
        _, partial = run.model.tokenizer.quantize(run.model.tokenizer.latent(target))
        restored.append(run.model.tokenizer.decode(partial[-1]).argmax(1).cpu().numpy())
    restored = np.concatenate(restored)
checks = solution_checks(regions[validation_indices], restored)
display(
    HTML(
        f"Validation pixel accuracy: {100*np.mean(restored == target_tokens[validation_indices]):.2f}%<br>"
        f"Exact targets: {100*np.mean(np.all(restored == target_tokens[validation_indices], axis=(1,2))):.2f}%<br>"
        f"Valid solutions after reconstruction: {100*checks['solved'].mean():.2f}%"
    )
)

## 5. Stage II — predict the next level

Freeze the entire tokenizer. The generator receives only the accumulated reconstruction of **preceding** levels, resized to the next level, along with positions and the level index. Ground-truth codes for the current level are never provided as input.

$$
p_\theta(r_{1:K})=\prod_{k=1}^{K}\prod_i
p_\theta(r_k^i\mid\widehat z^{(k-1)}),
\qquad
\mathcal L_{\mathrm{NCM}}=\frac1K\sum_{k=1}^{K}
\operatorname{CE}_{\mathrm{mean}}(r_k,\mathrm{logits}_k).
$$

This is Equation (10) with $\alpha=1$: each scale receives equal weight. During training, some **conditioning codes** are replaced by random codes from the same codebook; the replacement probability increases linearly from 0 to 0.1 across levels. Labels remain clean. This is a local hyperparameter choice, not the paper's Sudoku setting.

All levels are predicted in one packed pass with a block-diagonal target mask. The prefix sees only itself; each target block sees the prefix and its own level. Gradients from all levels and microbatches accumulate before **one** AdamW update. During generation, levels are sampled sequentially from the model's own predictions.


In conditional generation, each target block also sees the unchanged prefix of the original coloring. The prefix is not part of the quantized target and never contains queen positions.


In [9]:
import importlib
import _shared.ncm_training as ncm_training

train_online = importlib.reload(ncm_training).train_online

if run.state["tokenizer"]["updates"] < run.plan["tokenizer_updates"]:
    raise RuntimeError(
        "Complete the full tokenizer training cell before starting the generator."
    )

tokenizer_quality = run.tokenizer_quality(validation_dataset, examples=128)
if tokenizer_quality["exact_accuracy"] < TOKENIZER_MIN_EXACT:
    raise RuntimeError(
        "Tokenizer reconstruction is not ready for generation. Increase TOKENIZER_TARGET_UPDATES, "
        "rerun the targets cell, then continue stage I before training the generator."
    )
train_online(run, "generator", train_dataset, validation_dataset, report_every=100)

### 5.1 Checkpoint and audit

Save both stages, both optimizer states, the split, dataset fingerprint, RNG state, and actual training cost. NCM uses separate checkpoints from DFM and D3PM.


In [10]:
run.save()
audit = run.report()
actual_seconds = sum(s["seconds"] for s in run.state.values())
display(
    HTML(
        f"<b>One NCM run</b> · tokenizer + generator: {audit['parameters']['total']:,} parameters<br>"
        f"Actual total training: {actual_seconds/60:.2f} min · expanded update budget<br>"
        + "<br>".join(
            f"{name}: {s['updates']:,} / {run.plan[name+'_updates']:,} updates, "
            f"{s['examples']:,} examples, {s['seconds']/60:.2f} min"
            for name, s in run.state.items()
        )
        + f"<br>Checkpoint: <code>{html.escape(str(run.path.relative_to(PROJECT_ROOT)))}</code>"
    )
)

# A scalar mean can hide unlearned fine levels. Inspect validation CE by scale.
level_history = [
    h for h in run.history["generator"] if h.get("validation_level_ce") is not None
]
if level_history:
    level_fig = go.Figure()
    for k, size in enumerate(config.scales):
        level_fig.add_scatter(
            x=[h["update"] for h in level_history],
            y=[h["validation_level_ce"][k] for h in level_history],
            name=f"Level {k+1} · {size}×{size}",
            mode="lines+markers",
        )
    level_fig.update_layout(
        template="plotly_white",
        title="NCM · validation CE by level",
        xaxis_title="Generator updates",
        yaxis_title="Code cross-entropy (nats)",
        height=430,
        legend=dict(orientation="h", y=1.15),
        margin=dict(t=110),
    )
    level_fig.show(config=PLOT_CONFIG)

## 6. Solution trajectories

Start from $\widehat z^{(0)}=0$; the original coloring remains the prefix condition. At each level, sample categorical codes independently across positions, then add their contribution to the reconstruction. The final output uses decoder argmax.

Only the **original cell color or EMPTY** is allowed at the output, as in DFM. This restricts the vocabulary without enforcing 7 queens or repairing rows, columns, or colors. Evaluation does not select a successful attempt. Temperature **1.0** is fixed in advance. Previews decode each level; standard evaluation decodes only the final level.


In [11]:
run.require_trained()
GENERATION_TEMPERATURE = 1.0
for number, index in enumerate(splits["test"][:3]):
    _, frames = run.model.sample(
        coloring=regions[int(index)],
        temperature=GENERATION_TEMPERATURE,
        seed=SEED + 300 + number,
        capture=True,
    )
    panels = [
        regions[index],
        *[frame[0].numpy() for frame in frames],
        target_tokens[index],
    ]
    board_panels(
        panels,
        ["Puzzle", *[f"Level {i+1}" for i in range(len(frames))], "Target"],
        f"NCM · test puzzle {int(index)} · one attempt",
        colorings=[regions[index]] * len(panels),
        columns=4,
    ).show(config=PLOT_CONFIG)

## 7. Solvability on the same test puzzles

Puzzle-selection seed and order match DFM. Each coloring receives **32 independent attempts**, without solution search or answer repair. Success is checked against the rules, not just equality with the stored target. The 95% interval bootstraps whole puzzles rather than individual attempts.

Measure single-attempt success, the probability of at least one solution within $k$ attempts, and the distribution of puzzle difficulty. NCM uses 6 generator NFE + 1 decoder pass, compared with 100 U-Net NFE for DFM; equal attempt counts do not imply equal inference time, so timing is recorded separately.


In [12]:
N_EVAL_PUZZLES = 256
ATTEMPTS_PER_PUZZLE = 32
EVAL_BATCH_SIZE = 128
EVAL_SEED = SEED + 10_000
evaluation_indices = np.random.default_rng(EVAL_SEED).permutation(splits["test"])[
    :N_EVAL_PUZZLES
]
successes, predicted_tokens, evaluation = evaluate_queens(
    run,
    puzzles,
    evaluation_indices,
    attempts=ATTEMPTS_PER_PUZZLE,
    batch_size=EVAL_BATCH_SIZE,
    temperature=GENERATION_TEMPERATURE,
    seed=EVAL_SEED,
    artifact_dir=ARTIFACTS,
)
per_puzzle_success = successes.mean(1)
bootstrap_rng = np.random.default_rng(EVAL_SEED + 1)
bootstrap_means = per_puzzle_success[
    bootstrap_rng.integers(len(successes), size=(3000, len(successes)))
].mean(1)
low, high = np.quantile(bootstrap_means, [0.025, 0.975])
display(
    HTML(
        f"<b>NCM solved: {100*successes.mean():.2f}%</b> · 95% puzzle-bootstrap [{100*low:.2f}%, {100*high:.2f}%]<br>"
        f"At least once / {ATTEMPTS_PER_PUZZLE}: {100*successes.any(1).mean():.2f}%<br>"
        f"{len(successes)} puzzles · {successes.size:,} attempts · {evaluation['generation_seconds']:.2f} s"
    )
)
retry_fig = plot_queens_retries({"NCM": successes})
retry_fig.write_html(ARTIFACTS / "retry_curves.html", include_plotlyjs="cdn")
retry_fig.show(config=PLOT_CONFIG)

IntProgress(value=0, description='Solve Queens', layout=Layout(width='100%'), max=64, style=ProgressStyle(desc…

### 7.1 Solvability study

$\widehat p_i=c_i/32$ estimates the probability of solving an individual puzzle. Smooth the distribution with reflected Gaussian kernels of bandwidth $1/32$ on $[0,1]$, without hard bins. The vertical axis shows the percentage of puzzles per interval one percentage point wide, on a log scale. Boundary peaks at 0/32 and 32/32 remain visible.

This is a descriptive estimate from a finite number of attempts: zero observed successes do not establish zero true success probability.


In [13]:
grid = np.linspace(0, 1, 501)
bandwidth = 1 / ATTEMPTS_PER_PUZZLE
centers = np.concatenate(
    (per_puzzle_success, -per_puzzle_success, 2 - per_puzzle_success)
)
density = np.exp(-0.5 * ((grid[:, None] - centers[None, :]) / bandwidth) ** 2).sum(1)
density /= len(per_puzzle_success) * bandwidth * np.sqrt(2 * np.pi)
density /= np.trapezoid(density, grid)
study_fig = go.Figure(
    go.Scatter(
        x=100 * grid,
        y=np.maximum(density, 1e-6),
        mode="lines",
        line=dict(color="#4969ed", width=3),
        fill="tozeroy",
    )
)
study_fig.update_layout(
    template="plotly_white",
    title="NCM · solvability study",
    height=430,
    xaxis_title="Probability to be solved (%)",
    yaxis_title="Puzzles (% per 1 percentage point)",
    yaxis_type="log",
)
study_fig.write_html(ARTIFACTS / "solvability_study.html", include_plotlyjs="cdn")
study_fig.show(config=PLOT_CONFIG)
selected, target_p = [], float(per_puzzle_success.max())
for rank in range(5):
    candidates = [i for i in range(len(per_puzzle_success)) if i not in selected]
    position = min(candidates, key=lambda i: abs(per_puzzle_success[i] - target_p))
    selected.append(position)
    target_p = float(per_puzzle_success[position]) / 2
indices = evaluation_indices[selected]
board_panels(
    regions[indices],
    [
        f"{100*per_puzzle_success[p]:.2f}% · {successes[p].sum()}/{ATTEMPTS_PER_PUZZLE}"
        for p in selected
    ],
    "NCM · examples by solve probability",
    colorings=regions[indices],
    queens=True,
    queen_masks=queen_masks[indices],
    columns=5,
).show(config=PLOT_CONFIG)

## 8. NCM vs trained DFM

Load the **saved DFM** without changing its weights. Check the dataset SHA and test split, then evaluate the same puzzle IDs with 32 attempts. Cache entries are tied to checkpoint SHA and sampling settings.

One plot overlays both empirical retry curves with puzzle-bootstrap bands; dotted lines show $1-(1-\bar p)^k$ for each model. This approximation assumes equal difficulty across puzzles and can differ from averaging puzzle-specific probabilities. The scatter compares the two models' success probabilities on **the same coloring**.

NCM uses an expanded training budget; actual updates, time, and data exposure for both stages are recorded in the budget JSON. This comparison does not claim equal training compute. Quality is compared at equal attempt counts, with potentially different inference FLOPs. Test outcomes are never used for training or checkpoint selection.


In [14]:
if not DFM_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "A trained queens7_discrete_flow_matching.pt checkpoint is required for comparison."
    )
dfm, dfm_info = load_queens_dfm(
    DFM_CHECKPOINT_PATH,
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    device=DEVICE,
)
if not np.all(np.isin(evaluation_indices, dfm_info["test_indices"])):
    raise ValueError("Evaluated NCM puzzles are not all held out by DFM.")
comparison_signature = dict(
    dfm_sha256=dfm_info["checkpoint_sha256"],
    indices=evaluation_indices.tolist(),
    attempts=ATTEMPTS_PER_PUZZLE,
    steps=100,
    seed=EVAL_SEED,
    batch_size=EVAL_BATCH_SIZE,
    sampler="queens7_dfm_sampler_v1",
)
cache = ARTIFACTS / "dfm_comparison.npz"
cache_meta = ARTIFACTS / "dfm_comparison.json"
cached = (
    cache.exists()
    and cache_meta.exists()
    and json.loads(cache_meta.read_text(encoding="utf-8")) == comparison_signature
)
if cached:
    with np.load(cache, allow_pickle=False) as arrays:
        dfm_successes = arrays["successes"].copy()
else:
    boards = np.repeat(regions[evaluation_indices], ATTEMPTS_PER_PUZZLE, axis=0)
    progress = progress_bar("DFM comparison", math.ceil(len(boards) / EVAL_BATCH_SIZE))
    solved_batches = []
    for start in range(0, len(boards), EVAL_BATCH_SIZE):
        batch = boards[start : start + EVAL_BATCH_SIZE]
        tokens = sample_queens_dfm(
            dfm, batch, steps=100, seed=EVAL_SEED + start
        ).numpy()
        solved_batches.append(solution_checks(batch, tokens)["solved"])
        progress.value += 1
    progress.bar_style = "success"
    dfm_successes = np.concatenate(solved_batches).reshape(successes.shape)
    np.savez_compressed(cache, successes=dfm_successes)
    cache_meta.write_text(json.dumps(comparison_signature, indent=2), encoding="utf-8")
del dfm
comparison_fig = plot_queens_retries({"NCM": successes, "DFM": dfm_successes})
comparison_fig.write_html(ARTIFACTS / "ncm_vs_dfm_retries.html", include_plotlyjs="cdn")
comparison_fig.show(config=PLOT_CONFIG)
scatter = go.Figure()
for source in sorted(set(puzzles["source_names"][evaluation_indices])):
    mask = puzzles["source_names"][evaluation_indices] == source
    scatter.add_scatter(
        x=100 * dfm_successes[mask].mean(1),
        y=100 * successes[mask].mean(1),
        mode="markers",
        name=source,
        marker=dict(size=8, opacity=0.65),
        text=puzzles["source_ids"][evaluation_indices[mask]],
        hovertemplate="%{text}<br>DFM %{x:.2f}%<br>NCM %{y:.2f}%<extra></extra>",
    )
scatter.add_scatter(
    x=[0, 100],
    y=[0, 100],
    mode="lines",
    line=dict(color="#94a3b8", dash="dash"),
    showlegend=False,
    hoverinfo="skip",
)
scatter.update_layout(
    template="plotly_white",
    title="Same test puzzles · NCM vs DFM",
    height=550,
    xaxis=dict(title="DFM solvability (%)", range=[-2, 102]),
    yaxis=dict(
        title="NCM solvability (%)", range=[-2, 102], scaleanchor="x", scaleratio=1
    ),
)
scatter.write_html(ARTIFACTS / "ncm_vs_dfm_scatter.html", include_plotlyjs="cdn")
scatter.show(config=PLOT_CONFIG)
comparison_rows = np.column_stack(
    (evaluation_indices, dfm_successes.mean(1), successes.mean(1))
)
np.savetxt(
    ARTIFACTS / "ncm_vs_dfm.csv",
    comparison_rows,
    delimiter=",",
    comments="",
    header="puzzle_index,dfm_probability,ncm_probability",
    fmt=["%d", "%.8f", "%.8f"],
)
display(
    HTML(
        f"DFM: {100*dfm_successes.mean():.2f}% · {dfm_info['global_step']:,} train updates<br>"
        f"NCM: {100*successes.mean():.2f}% · "
        + " + ".join(
            f"{stage} {s['updates']:,} updates" for stage, s in run.state.items()
        )
    )
)

IntProgress(value=0, description='DFM comparison', layout=Layout(width='100%'), max=64, style=ProgressStyle(de…